In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!pip install mne

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 59.0 MB/s eta 0:00:00


In [ ]:
import os
import glob, mne
import csv

# get all filepaths

base_path = "/content/drive/MyDrive/MEng/6.S985/ds005873/sub-"
eeg_middle_part = "/ses-01/eeg/sub-"
ecg_middle_part = "/ses-01/ecg/sub-"
common_task_prefix = "_ses-01_task-szMonitoring_"
eeg_file_extension = "_eeg.edf"
ecg_file_extension = "_ecg.edf"

# all_eeg_paths = []
# all_ecg_paths = []

# for i in range(1, 126):
#   pt_num = str(i).zfill(3)

#   # Construct EEG path pattern with wildcard for run number
#   eeg_pattern = os.path.join(
#       base_path + pt_num + eeg_middle_part + pt_num + common_task_prefix + 'run-*' + eeg_file_extension
#   )
#   new_eeg = glob.glob(eeg_pattern)
#   # Construct ECG path pattern with wildcard for run number
#   ecg_pattern = os.path.join(
#       base_path + pt_num + ecg_middle_part + pt_num + common_task_prefix + 'run-*' + ecg_file_extension
#   )
#   new_ecg = glob.glob(ecg_pattern)

#   # only add EEG/ECG if both are present (len of new lists are same)
#   if len(new_eeg)==len(new_ecg):
#     # Find all matching EEG files
#     all_eeg_paths.extend(new_eeg)
#     # Find all matching ECG files
#     all_ecg_paths.extend(new_ecg)
#   else: # only add runs that have both eeg and ecg
#     for run in new_eeg:
#       ecg_equiv = run.replace(eeg_middle_part, ecg_middle_part).replace(eeg_file_extension, ecg_file_extension)
#       if ecg_equiv in new_ecg:
#         all_eeg_paths.append(run)
#         all_ecg_paths.append(ecg_equiv)

# print(f"Total EEG paths collected: {len(all_eeg_paths)}")
# print(f"Total ECG paths collected: {len(all_ecg_paths)}")

# print("\nFirst 5 EEG paths (dynamic runs):")
# for path in all_eeg_paths[:5]:
#     print(path)

# print("\nFirst 5 ECG paths (dynamic runs):")
# for path in all_ecg_paths[:5]:
#     print(path)


# f = open(r'/content/drive/MyDrive/MEng/6.S985/ds005873/all_eeg_paths.csv', 'w', newline='')
# # Join list items with commas and write as a single string
# f.write(",".join(all_eeg_paths))
# f.close()

# g = open(r'/content/drive/MyDrive/MEng/6.S985/ds005873/all_ecg_paths.csv', 'w', newline='')
# g.write(",".join(all_ecg_paths))
# g.close()

f = open(r'/content/drive/MyDrive/MEng/6.S985/ds005873/all_eeg_paths.csv', 'r')
all_eeg_paths = f.read().split(',')
f.close()

g = open(r'/content/drive/MyDrive/MEng/6.S985/ds005873/all_ecg_paths.csv', 'r')
all_ecg_paths = g.read().split(',')
g.close()

# 2804 EEG paths total

In [ ]:
import os, json, re
import pandas as pd
import numpy as np

EVENTS_JSON = "/content/drive/MyDrive/MEng/6.S985/ds005873/events.json"

def eeg_path_to_events_path(eeg_path: str) -> str:
    """
    Converts .../ses-01/eeg/<stem>_eeg.edf  ->  .../ses-01/eeg/<stem>_events.tsv
    where <stem> = sub-XXX_ses-01_task-..._run-YY
    """
    events_path = eeg_path.replace("_eeg.edf", "_events.tsv")
    return events_path

def parse_ids_from_path(eeg_path):
    filename = os.path.basename(eeg_path).replace("_eeg.edf", "")
    parts = filename.split("_")
    patient_id = parts[0]          # sub-001
    filename_parts = filename.split("_")
    run_id = "_".join([filename_parts[0], filename_parts[-1]]) # sub-001_run-03
    return patient_id, run_id

# returns dictionary of {'bckg': description of bckg, etc. for all event types}
def load_eventtype_levels(events_json_path=EVENTS_JSON):
    """
    Returns dict like {"bckg": "...", "impd": "...", "sz_foc": "...", ...}
    """
    with open(events_json_path, "r") as f:
        meta = json.load(f)
    return meta.get("eventType", {}).get("Levels", {})

# sz_types is set of sz types, non_sz is non_sz set (impd and bckg)
def build_sz_and_nonsz_sets(eventtype_levels):
    """
    Rule from your events.json:
      - non-seizure: bckg, impd
      - seizure: everything else in Levels
    """
    non_sz = {"bckg", "impd"}
    all_types = set(eventtype_levels.keys())
    sz_types = all_types - non_sz
    return sz_types, non_sz

SZ_TYPES, NON_SZ_TYPES = build_sz_and_nonsz_sets(load_eventtype_levels(EVENTS_JSON))
# print("Non-seizure types:", NON_SZ_TYPES)
# print("Example seizure types:", sorted(list(SZ_TYPES))[:10])

def eeg_path_to_events_path(eeg_path):
    """
    ds005873 structure:
      .../sub-XXX/ses-YY/eeg/<stem>_eeg.edf
    events file:
      .../sub-XXX/ses-YY/eeg/<stem>_events.tsv
    """
    # swap _eeg.edf -> _events.tsv
    return eeg_path.replace("_eeg.edf", "_events.tsv")

def load_events_tsv(events_path):
    """
    Load events TSV, coerce onset/duration numeric, add end column.
    """
    ev = pd.read_csv(events_path, sep="\t")
    ev["onset"] = pd.to_numeric(ev["onset"], errors="coerce")
    ev["duration"] = pd.to_numeric(ev["duration"], errors="coerce")
    ev = ev.dropna(subset=["onset", "duration"])
    ev["end"] = ev["onset"] + ev["duration"]
    ev["eventType"] = ev["eventType"].astype(str).str.strip()
    ev['lateralization'] = ev['lateralization'].astype(str).str.strip()
    ev['localization'] = ev['localization'].astype(str).str.strip()
    ev['vigilance'] = ev['vigilance'].astype(str).str.strip()
    return ev

# returns something like ['impd', 'impd'] if 2 impd events occured in time window
def overlapping_eventtypes(ev, t0_sec, window_sec):
    """
    Return list of eventTypes whose [onset,end] overlaps [t0, t1].
    Overlap condition: onset < t1 and end > t0
    """
    t0 = float(t0_sec)
    t1 = t0 + float(window_sec)
    m = (ev["onset"] < t1) & (ev["end"] > t0)
    ev_overlaps = ev[m]
    sz = (ev_overlaps['eventType'] != 'impd') & (ev_overlaps['eventType'] != 'bckg')
    sz_ev_overlaps = ev_overlaps[sz]
    return sz_ev_overlaps

# calls load_events_tsv and overlapping_eventtypes
def label_window_from_events(events_path=None, events_df=None, t0_sec=0, window_sec=10, sz_types=SZ_TYPES, non_sz_types=NON_SZ_TYPES):
    """
    Label seizure if ANY overlapping eventType is in sz_types.
    Otherwise nonseizure.

    Returns a dictionary with 'label' and other relevant seizure event details.
    """
    if events_df is not None:
        ev = events_df
        n_events_in_file = len(events_df)
    elif events_path and os.path.exists(events_path):
        ev = load_events_tsv(events_path)
        n_events_in_file = len(ev)
    else:
        raise ValueError("Either events_df or events_path must be provided")

    overlapped = overlapping_eventtypes(ev, t0_sec, window_sec)
    if not overlapped.empty:
        first_sz_overlap = overlapped.iloc[0] # Take the first overlapping seizure event if multiple
        return {
            'label': first_sz_overlap['eventType'],
            'binary_label': 0 if first_sz_overlap['eventType'] in non_sz_types else 1,
            'seizure_onset_sec': first_sz_overlap['onset'], # Onset relative to run start
            'seizure_duration_sec': first_sz_overlap['duration'],
            'lateralization': first_sz_overlap['lateralization'],
            'localization': first_sz_overlap['localization'],
            'vigilance': first_sz_overlap['vigilance']
        }
    else:
        return {
            'label': 'nonseizure',
            'binary_label': 0,
            'seizure_onset_sec': np.nan,
            'seizure_duration_sec': np.nan,
            'lateralization': 'nonseizure', # Placeholder for categorical if no seizure
            'localization': 'nonseizure',
            'vigilance': 'nonseizure'
        }

eeg_path = "/content/drive/MyDrive/MEng/6.S985/ds005873/sub-001/ses-01/eeg/sub-001_ses-01_task-szMonitoring_run-03_eeg.edf"
# events_path = eeg_path_to_events_path(eeg_path)
print(parse_ids_from_path(eeg_path))
# ev = load_events_tsv(events_path)
# label = label_window_from_events(events_path=events_path, t0_sec=23059, window_sec=10)
# print(label)


('sub-001', 'sub-001_run-03')


In [ ]:
# make sz and non-sz type set
EVENTS_JSON = "/content/drive/MyDrive/MEng/6.S985/ds005873/events.json"

levels = load_eventtype_levels(EVENTS_JSON)
sz_types, non_sz_types = build_sz_and_nonsz_sets(levels)

print("Non-seizure types:", non_sz_types)
print("Example seizure types:", list(sorted(sz_types))[:10])

Non-seizure types: {'bckg', 'impd'}
Example seizure types: ['sz', 'sz_foc', 'sz_foc_a', 'sz_foc_a_m', 'sz_foc_a_m_atonic', 'sz_foc_a_m_automatisms', 'sz_foc_a_m_clonic', 'sz_foc_a_m_hyperkinetic', 'sz_foc_a_m_myoclonic', 'sz_foc_a_m_spasms']


In [ ]:
# load .edf windows for both EEG and ECG

import numpy as np
import mne
import pandas as pd

def load_edf_windows(
    eeg_edf_path: str,
    ecg_edf_path: str,
    picks=None,                 # list of channel names OR None to infer
    window_sec: float = 10.0,
    overlap_sec: float = 5.0,
    resample_hz: float | None = 256.0,   # set None to keep original
    preload: bool = True,
    verbose: bool = False
):
    patient_id, run_id = parse_ids_from_path(eeg_edf_path)

    raw_eeg = mne.io.read_raw_edf(eeg_edf_path, preload=preload, verbose=verbose)
    raw_ecg = mne.io.read_raw_edf(ecg_edf_path, preload=preload, verbose=verbose)

    # If channel picks not provided, keep all channels
    if picks is None:
        picks_idx = mne.pick_types(raw_eeg.info, eeg=True, ecg=True, emg=True, misc=True, stim=False, exclude=[])
    else:
        picks_idx = mne.pick_channels(raw_eeg.ch_names, include=picks)

    raw_pick_eeg = raw_eeg.copy().pick(picks_idx)
    raw_pick_ecg = raw_ecg.copy()

    # Resample for consistency (recommended for feature extraction)
    if resample_hz is not None:
        raw_pick_eeg.resample(resample_hz)
        raw_pick_ecg.resample(resample_hz)

    eeg_data_full_run = raw_pick_eeg.get_data()  # shape: (n_channels, n_timepoints)
    ecg_data_full_run = raw_pick_ecg.get_data()  # shape: (n_channels, n_timepoints)
    sfreq = raw_pick_eeg.info["sfreq"]
    ch_names_eeg = raw_pick_eeg.ch_names
    ch_names_ecg = raw_pick_ecg.ch_names

    win_samp = int(window_sec * sfreq)
    hop_samp = int((window_sec - overlap_sec) * sfreq)
    if hop_samp <= 0:
        raise ValueError("overlap_sec must be < window_sec")

    n_ch_eeg, n_t_eeg = eeg_data_full_run.shape
    # Use the minimum length between EEG and ECG to avoid indexing errors
    # In a real scenario, you might want to handle length mismatches more robustly
    n_t = min(n_t_eeg, ecg_data_full_run.shape[1])

    starts = np.arange(0, n_t - win_samp + 1, hop_samp)

    # get events for this specific patient's run
    events_path = eeg_path_to_events_path(eeg_edf_path)
    events_tsv = load_events_tsv(events_path)

    all_window_data = []
    for s in starts:
      eeg_window_data = eeg_data_full_run[:, s:s+win_samp]
      ecg_window_data = ecg_data_full_run[:, s:s+win_samp]
      start_time = s/sfreq
      end_time = s/sfreq + window_sec

      # Label the window and get all metadata
      metadata_info = label_window_from_events(
          events_df=events_tsv, # Pass the pre-loaded DataFrame
          t0_sec=start_time,
          window_sec=window_sec
      )

      X = {
          'patient_id': patient_id,
          'run_id': run_id,
          'eeg_data': eeg_window_data, # Store the actual window's EEG data
          'ecg_data': ecg_window_data, # Store the actual window's ECG data
          'start_time': start_time,
          'end_time': end_time,
          'sfreq': sfreq,
          'eeg_channels': ch_names_eeg,
          'ecg_channels': ch_names_ecg,
          **metadata_info # Unpack the metadata dictionary directly
           }

      all_window_data.append(X)

    # Return the list of window dictionaries
    return all_window_data


In [ ]:
import os
import json
import h5py
import numpy as np
import pandas as pd

def save_windowed_run(all_window_data, output_prefix):
    eeg_windows = np.stack([w["eeg_data"] for w in all_window_data], axis=0)
    ecg_windows = np.stack([w["ecg_data"] for w in all_window_data], axis=0)

    metadata_rows = []
    for i, w in enumerate(all_window_data):
        metadata_rows.append({
            "patient_id": w['patient_id'],
            'run_id': w['run_id'],
            "window_idx": i,
            "start_time": w["start_time"],
            "end_time": w["end_time"],
            "label": w["label"],
            "binary_label": w.get("binary_label"),
            "lateralization": w.get("lateralization"),
            "localization": w.get("localization"),
            "vigilance": w.get("vigilance"),
            "seizure_onset_sec": w.get("seizure_onset_sec"),
            "seizure_duration_sec": w.get("seizure_duration_sec"),
        })

    metadata_df = pd.DataFrame(metadata_rows)

    h5_path = output_prefix + ".h5"
    meta_path = output_prefix + "_metadata.parquet"
    info_path = output_prefix + "_info.json"

    with h5py.File(h5_path, "w") as f:
        f.create_dataset("eeg_windows", data=eeg_windows, compression="gzip")
        f.create_dataset("ecg_windows", data=ecg_windows, compression="gzip")

    metadata_df.to_parquet(meta_path, index=False)

    run_info = {
        "sfreq": all_window_data[0]["sfreq"],
        "eeg_channels": all_window_data[0]["eeg_channels"],
        "ecg_channels": all_window_data[0]["ecg_channels"],
        "num_windows": len(all_window_data),
        "run_id": all_window_data[0]["run_id"],
        "patient_id": all_window_data[0]["patient_id"],
    }
    with open(info_path, "w") as fp:
        json.dump(run_info, fp, indent=2)

In [ ]:
eeg_path = all_eeg_paths[2]
ecg_path = all_ecg_paths[2]

all_window_data = load_edf_windows(eeg_path, ecg_path)

print("num windows:", len(all_window_data))
print("first eeg shape:", all_window_data[0]["eeg_data"].shape)
print("first ecg shape:", all_window_data[0]["ecg_data"].shape)
print("first label:", all_window_data[11595]["label"])
print("first patient_id:", all_window_data[0]["patient_id"])
print("first run_id:", all_window_data[0]["run_id"])

Sampling frequency of the instance is already 256.0, returning unmodified.
Sampling frequency of the instance is already 256.0, returning unmodified.
num windows: 12994
first eeg shape: (2, 2560)
first ecg shape: (1, 2560)
first label: sz_foc_ia_nm
first patient_id: sub-001
first run_id: sub-001_run-03


In [ ]:
output_prefix = "/content/drive/MyDrive/MEng/6.S985/ds005873/processed_windows/sub-001_ses-01_task-szMonitoring_run-03"
save_windowed_run(all_window_data, output_prefix)

In [ ]:
import h5py
import pandas as pd
import json

with h5py.File(output_prefix + ".h5", "r") as f:
    print("EEG shape:", f["eeg_windows"].shape)
    print("ECG shape:", f["ecg_windows"].shape)

meta = pd.read_parquet(output_prefix + "_metadata.parquet")
print(meta.head())
print(meta.columns)

with open(output_prefix + "_info.json", "r") as fp:
    run_info = json.load(fp)
print(run_info)

EEG shape: (12994, 2, 2560)
ECG shape: (12994, 1, 2560)
  patient_id          run_id  window_idx  start_time  end_time       label  \
0    sub-001  sub-001_run-03           0         0.0      10.0  nonseizure   
1    sub-001  sub-001_run-03           1         5.0      15.0  nonseizure   
2    sub-001  sub-001_run-03           2        10.0      20.0  nonseizure   
3    sub-001  sub-001_run-03           3        15.0      25.0  nonseizure   
4    sub-001  sub-001_run-03           4        20.0      30.0  nonseizure   

   binary_label lateralization localization   vigilance  seizure_onset_sec  \
0             0     nonseizure   nonseizure  nonseizure                NaN   
1             0     nonseizure   nonseizure  nonseizure                NaN   
2             0     nonseizure   nonseizure  nonseizure                NaN   
3             0     nonseizure   nonseizure  nonseizure                NaN   
4             0     nonseizure   nonseizure  nonseizure                NaN   

   sei

In [ ]:
n_seizure = sum(w["binary_label"] == 1 for w in all_window_data)
n_nonseizure = sum(w["binary_label"] == 0 for w in all_window_data)

print("num windows:", len(all_window_data))
print("num seizure:", n_seizure)
print("num nonseizure:", n_nonseizure)

num windows: 12994
num seizure: 16
num nonseizure: 12978


In [ ]:
print(all_eeg_paths[0].split('/'))

['', 'content', 'drive', 'MyDrive', 'MEng', '6.S985', 'ds005873', 'sub-001', 'ses-01', 'eeg', 'sub-001_ses-01_task-szMonitoring_run-01_eeg.edf']


In [ ]:
# old script that doesn't filter out completely nonseizure runs
# or balance 1:4 ratio of sz to non-sz

# import os, random

# start = 0
# end = 1000

# for i, (eeg_path, ecg_path) in enumerate(zip(all_eeg_paths[start:end], all_ecg_paths[start:end])):
#     try:
#         end_path = eeg_path.replace("_eeg.edf", "").split("/")[-1]
#         output_prefix = "/content/drive/MyDrive/MEng/6.S985/ds005873/processed_windows_2sec/" + end_path

#         h5_path = output_prefix + ".h5"
#         meta_path = output_prefix + "_metadata.parquet"
#         info_path = output_prefix + "_info.json"

#         if os.path.exists(h5_path) and os.path.exists(meta_path) and os.path.exists(info_path):
#             print(i, "SKIPPING already processed:", output_prefix)
#             continue

#         # add a check to see if run doesn't have seizure instance

#         all_window_data = load_edf_windows(eeg_path, ecg_path, window_sec=2, overlap_sec=1)
#         save_windowed_run(all_window_data, output_prefix)

#         n_seizure = sum(w["binary_label"] == 1 for w in all_window_data)
#         n_nonseizure = sum(w["binary_label"] == 0 for w in all_window_data)

#         print(i, "saved", output_prefix, "windows:", len(all_window_data),
#               "seizure:", n_seizure, "nonseizure:", n_nonseizure)

#     except Exception as e:
#         print(i, "FAILED", eeg_path, str(e))

Sampling frequency of the instance is already 256.0, returning unmodified.
Sampling frequency of the instance is already 256.0, returning unmodified.
0 saved /content/drive/MyDrive/MEng/6.S985/ds005873/processed_windows/sub-028_ses-01_task-szMonitoring_run-01 windows: 1189 seizure: 7 nonseizure: 1182
Sampling frequency of the instance is already 256.0, returning unmodified.
Sampling frequency of the instance is already 256.0, returning unmodified.
1 saved /content/drive/MyDrive/MEng/6.S985/ds005873/processed_windows/sub-028_ses-01_task-szMonitoring_run-02 windows: 12408 seizure: 0 nonseizure: 12408
Sampling frequency of the instance is already 256.0, returning unmodified.
Sampling frequency of the instance is already 256.0, returning unmodified.
2 saved /content/drive/MyDrive/MEng/6.S985/ds005873/processed_windows/sub-028_ses-01_task-szMonitoring_run-03 windows: 2639 seizure: 0 nonseizure: 2639
Sampling frequency of the instance is already 256.0, returning unmodified.
Sampling frequenc

In [21]:
import os
import random
import pandas as pd

random.seed(0)

# Define the new output directory
output_base_dir = "/content/drive/MyDrive/MEng/6.S985/ds005873/processed_windows_2sec_seizure_only"
os.makedirs(output_base_dir, exist_ok=True)

# 1. Extract unique patient IDs
patient_ids = set()
for path in all_eeg_paths:
    patient_id, _ = parse_ids_from_path(path)
    patient_ids.add(patient_id)

patient_ids = list(patient_ids)

# 2. Randomly select 50 patient IDs
if len(patient_ids) > 50:
    selected_patients = random.sample(patient_ids, 50)
else:
    selected_patients = patient_ids

print(f"Selected {len(selected_patients)} patients for processing: {selected_patients}")

# Helper function to check for seizure events in a run
def contains_seizure_events(events_df, non_sz_types_set):
    if events_df.empty: # If no events, no seizures
        return False
    # Check if any eventType is NOT in the non_sz_types_set
    return any(eventType not in non_sz_types_set for eventType in events_df['eventType'])

# 3. Iterate through all runs and process based on criteria
processed_count = 0
skipped_non_seizure_count = 0
skipped_existing_count = 0

for i, (eeg_path, ecg_path) in enumerate(zip(all_eeg_paths, all_ecg_paths)):
    patient_id, run_id = parse_ids_from_path(eeg_path)

    if patient_id not in selected_patients:
        continue # Skip if not one of the randomly selected patients

    try:
        # Construct the output prefix for this run
        filename_stem = os.path.basename(eeg_path).replace("_eeg.edf", "")
        output_prefix = os.path.join(output_base_dir, filename_stem)

        h5_path = output_prefix + ".h5"
        meta_path = output_prefix + "_metadata.parquet"
        info_path = output_prefix + "_info.json"

        # Check if the run has already been processed and saved
        if os.path.exists(h5_path) and os.path.exists(meta_path) and os.path.exists(info_path):
            print(f"{i+1}/{len(all_eeg_paths)} SKIPPING existing processed run: {filename_stem}")
            skipped_existing_count += 1
            continue

        # Determine if the run contains seizure events
        events_path = eeg_path_to_events_path(eeg_path)
        # Ensure events_path exists before trying to load it
        if not os.path.exists(events_path):
            print(f"{i+1}/{len(all_eeg_paths)} FAILED to find events file for {filename_stem}. Skipping this run.")
            continue

        events_df = load_events_tsv(events_path)

        if not contains_seizure_events(events_df, NON_SZ_TYPES):
            print(f"{i+1}/{len(all_eeg_paths)} SKIPPING run with no seizure events: {filename_stem}")
            skipped_non_seizure_count += 1
            continue

        # If we reach here, the run contains seizure events and needs processing
        print(f"{i+1}/{len(all_eeg_paths)} PROCESSING run with seizure events: {filename_stem}")
        all_window_data = load_edf_windows(eeg_path, ecg_path, window_sec=2, overlap_sec=1)

        # Only save if there's data to save
        if all_window_data:
            save_windowed_run(all_window_data, output_prefix)
            n_seizure = sum(w["binary_label"] == 1 for w in all_window_data)
            n_nonseizure = sum(w["binary_label"] == 0 for w in all_window_data)

            print(f"  Saved {output_prefix}. Windows: {len(all_window_data)}, Seizure: {n_seizure}, Non-seizure: {n_nonseizure}")
            processed_count += 1
        else:
            print(f"  No window data generated for {filename_stem}. Skipping save.")

    except Exception as e:
        print(f"{i+1}/{len(all_eeg_paths)} FAILED to process {filename_stem}: {str(e)}")

print("\n--- Processing Summary ---")
print(f"Total runs processed: {processed_count}")
print(f"Total runs skipped (no seizure events): {skipped_non_seizure_count}")
print(f"Total runs skipped (already existing): {skipped_existing_count}")
print(f"Total patients considered: {len(selected_patients)}")


Selected 50 patients for processing: ['sub-093', 'sub-022', 'sub-059', 'sub-030', 'sub-005', 'sub-092', 'sub-062', 'sub-100', 'sub-114', 'sub-107', 'sub-006', 'sub-025', 'sub-074', 'sub-088', 'sub-007', 'sub-070', 'sub-073', 'sub-119', 'sub-115', 'sub-106', 'sub-090', 'sub-050', 'sub-123', 'sub-094', 'sub-113', 'sub-065', 'sub-020', 'sub-110', 'sub-121', 'sub-035', 'sub-109', 'sub-075', 'sub-079', 'sub-043', 'sub-013', 'sub-091', 'sub-071', 'sub-012', 'sub-032', 'sub-083', 'sub-011', 'sub-101', 'sub-089', 'sub-044', 'sub-018', 'sub-036', 'sub-118', 'sub-048', 'sub-054', 'sub-053']
33/2804 SKIPPING run with no seizure events: sub-005_ses-01_task-szMonitoring_run-01
34/2804 SKIPPING run with no seizure events: sub-005_ses-01_task-szMonitoring_run-02
35/2804 SKIPPING run with no seizure events: sub-005_ses-01_task-szMonitoring_run-03
36/2804 SKIPPING run with no seizure events: sub-005_ses-01_task-szMonitoring_run-04
37/2804 SKIPPING run with no seizure events: sub-005_ses-01_task-szMonit

KeyboardInterrupt: 

In [22]:
def balance_classes_by_sampling(all_window_data, seizure_ratio=1, nonseizure_ratio=5, random_seed=None):
    if random_seed is not None:
        random.seed(random_seed)

    seizure_windows = [w for w in all_window_data if w['binary_label'] == 1]
    nonseizure_windows = [w for w in all_window_data if w['binary_label'] == 0]

    num_seizure = len(seizure_windows)
    num_nonseizure_needed = num_seizure * nonseizure_ratio // seizure_ratio

    if num_nonseizure_needed == 0 and num_seizure == 0:
        # If no seizure events and no non-seizure events, return empty list
        return []
    elif num_nonseizure_needed == 0 and num_seizure > 0:
        # If there are seizure events but no non-seizure events are needed (e.g., nonseizure_ratio=0), return only seizure
        return seizure_windows
    elif num_seizure == 0 and num_nonseizure_needed > 0:
        # If no seizure events but non-seizure events are needed (this case shouldn't happen if previous logic works), return empty or all non-seizure
        # For this specific use case (processing runs *with* seizure events), this block might be unnecessary
        # If a run is passed to this function, it implies it has seizure events.
        # However, for robustness, if for some reason num_seizure is 0, we should handle it.
        # If we reach here, it means a run marked as containing seizure events by contains_seizure_events
        # actually produced no seizure windows after load_edf_windows. This is an inconsistency that might need investigation.
        # For now, we'll return an empty list if no seizure windows are found to base the sampling on.
        return []

    if len(nonseizure_windows) <= num_nonseizure_needed:
        # If not enough non-seizure windows, take all of them
        sampled_nonseizure_windows = nonseizure_windows
    else:
        sampled_nonseizure_windows = random.sample(nonseizure_windows, num_nonseizure_needed)

    balanced_windows = seizure_windows + sampled_nonseizure_windows
    random.shuffle(balanced_windows)

    return balanced_windows

In [ ]:
import os
import random
import pandas as pd

random.seed(0)
# Selected 50 patients for processing: ['sub-093', 'sub-022', 'sub-059', 'sub-030', 'sub-005', 'sub-092', 'sub-062', 'sub-100', 'sub-114', 'sub-107', 'sub-006', 'sub-025', 'sub-074', 'sub-088', 'sub-007', 'sub-070', 'sub-073', 'sub-119', 'sub-115', 'sub-106', 'sub-090', 'sub-050', 'sub-123', 'sub-094', 'sub-113', 'sub-065', 'sub-020', 'sub-110', 'sub-121', 'sub-035', 'sub-109', 'sub-075', 'sub-079', 'sub-043', 'sub-013', 'sub-091', 'sub-071', 'sub-012', 'sub-032', 'sub-083', 'sub-011', 'sub-101', 'sub-089', 'sub-044', 'sub-018', 'sub-036', 'sub-118', 'sub-048', 'sub-054', 'sub-053']

# Define the new output directory
output_base_dir = "/content/drive/MyDrive/MEng/6.S985/ds005873/processed_windows_2sec_seizure_only"
os.makedirs(output_base_dir, exist_ok=True)

# 1. Extract unique patient IDs
patient_ids = set()
for path in all_eeg_paths:
    patient_id, _ = parse_ids_from_path(path)
    patient_ids.add(patient_id)

patient_ids = list(patient_ids)

# 2. Randomly select 50 patient IDs
if len(patient_ids) > 50:
    selected_patients = random.sample(patient_ids, 50)
else:
    selected_patients = patient_ids

print(f"Selected {len(selected_patients)} patients for processing: {selected_patients}")

# Helper function to check for seizure events in a run
def contains_seizure_events(events_df, non_sz_types_set):
    if events_df.empty: # If no events, no seizures
        return False
    # Check if any eventType is NOT in the non_sz_types_set
    return any(eventType not in non_sz_types_set for eventType in events_df['eventType'])

# 3. Iterate through all runs and process based on criteria
processed_count = 0
skipped_non_seizure_count = 0
skipped_existing_count = 0

for i, (eeg_path, ecg_path) in enumerate(zip(all_eeg_paths, all_ecg_paths)):
    patient_id, run_id = parse_ids_from_path(eeg_path)

    if patient_id not in selected_patients:
        continue # Skip if not one of the randomly selected patients

    try:
        # Construct the output prefix for this run
        filename_stem = os.path.basename(eeg_path).replace("_eeg.edf", "")
        output_prefix = os.path.join(output_base_dir, filename_stem)

        h5_path = output_prefix + ".h5"
        meta_path = output_prefix + "_metadata.parquet"
        info_path = output_prefix + "_info.json"

        # Check if the run has already been processed and saved
        if os.path.exists(h5_path) and os.path.exists(meta_path) and os.path.exists(info_path):
            print(f"{i+1}/{len(all_eeg_paths)} SKIPPING existing processed run: {filename_stem}")
            skipped_existing_count += 1
            continue

        # Determine if the run contains seizure events
        events_path = eeg_path_to_events_path(eeg_path)
        # Ensure events_path exists before trying to load it
        if not os.path.exists(events_path):
            print(f"{i+1}/{len(all_eeg_paths)} FAILED to find events file for {filename_stem}. Skipping this run.")
            continue

        events_df = load_events_tsv(events_path)

        if not contains_seizure_events(events_df, NON_SZ_TYPES):
            print(f"{i+1}/{len(all_eeg_paths)} SKIPPING run with no seizure events: {filename_stem}")
            skipped_non_seizure_count += 1
            continue

        # If we reach here, the run contains seizure events and needs processing
        print(f"{i+1}/{len(all_eeg_paths)} PROCESSING run with seizure events: {filename_stem}")
        raw_all_window_data = load_edf_windows(eeg_path, ecg_path, window_sec=2, overlap_sec=1)

        # Apply class balancing
        all_window_data = balance_classes_by_sampling(raw_all_window_data, random_seed=0)

        # Only save if there's data to save
        if all_window_data:
            save_windowed_run(all_window_data, output_prefix)
            n_seizure = sum(w["binary_label"] == 1 for w in all_window_data)
            n_nonseizure = sum(w["binary_label"] == 0 for w in all_window_data)

            print(f"  Saved {output_prefix}. Windows: {len(all_window_data)}, Seizure: {n_seizure}, Non-seizure: {n_nonseizure}")
            processed_count += 1
        else:
            print(f"  No window data generated for {filename_stem}. Skipping save.")

    except Exception as e:
        print(f"{i+1}/{len(all_eeg_paths)} FAILED to process {filename_stem}: {str(e)}")

print("\n--- Processing Summary ---")
print(f"Total runs processed: {processed_count}")
print(f"Total runs skipped (no seizure events): {skipped_non_seizure_count}")
print(f"Total runs skipped (already existing): {skipped_existing_count}")
print(f"Total patients considered: {len(selected_patients)}")


Selected 50 patients for processing: ['sub-093', 'sub-022', 'sub-059', 'sub-030', 'sub-005', 'sub-092', 'sub-062', 'sub-100', 'sub-114', 'sub-107', 'sub-006', 'sub-025', 'sub-074', 'sub-088', 'sub-007', 'sub-070', 'sub-073', 'sub-119', 'sub-115', 'sub-106', 'sub-090', 'sub-050', 'sub-123', 'sub-094', 'sub-113', 'sub-065', 'sub-020', 'sub-110', 'sub-121', 'sub-035', 'sub-109', 'sub-075', 'sub-079', 'sub-043', 'sub-013', 'sub-091', 'sub-071', 'sub-012', 'sub-032', 'sub-083', 'sub-011', 'sub-101', 'sub-089', 'sub-044', 'sub-018', 'sub-036', 'sub-118', 'sub-048', 'sub-054', 'sub-053']
33/2804 SKIPPING run with no seizure events: sub-005_ses-01_task-szMonitoring_run-01
34/2804 SKIPPING run with no seizure events: sub-005_ses-01_task-szMonitoring_run-02
35/2804 SKIPPING run with no seizure events: sub-005_ses-01_task-szMonitoring_run-03
36/2804 SKIPPING run with no seizure events: sub-005_ses-01_task-szMonitoring_run-04
37/2804 SKIPPING run with no seizure events: sub-005_ses-01_task-szMonit

In [ ]:
import pandas as pd

output_prefix = "/content/drive/MyDrive/MEng/6.S985/ds005873/processed_windows/sub-001_ses-01_task-szMonitoring_run-03"

meta = pd.read_parquet(output_prefix + "_metadata.parquet")

sz_meta = meta[meta["binary_label"] == 1].copy()

print("num seizure windows:", len(sz_meta))
print(sz_meta.head(20))

num seizure windows: 16
      patient_id          run_id  window_idx  start_time  end_time  \
11594    sub-001  sub-001_run-03       11594     57970.0   57980.0   
11595    sub-001  sub-001_run-03       11595     57975.0   57985.0   
11596    sub-001  sub-001_run-03       11596     57980.0   57990.0   
11597    sub-001  sub-001_run-03       11597     57985.0   57995.0   
11598    sub-001  sub-001_run-03       11598     57990.0   58000.0   
11599    sub-001  sub-001_run-03       11599     57995.0   58005.0   
11600    sub-001  sub-001_run-03       11600     58000.0   58010.0   
11601    sub-001  sub-001_run-03       11601     58005.0   58015.0   
11602    sub-001  sub-001_run-03       11602     58010.0   58020.0   
11603    sub-001  sub-001_run-03       11603     58015.0   58025.0   
11604    sub-001  sub-001_run-03       11604     58020.0   58030.0   
11605    sub-001  sub-001_run-03       11605     58025.0   58035.0   
11606    sub-001  sub-001_run-03       11606     58030.0   58040.0

In [ ]:
cols = [
    "patient_id",
    "run_id",
    "window_idx",
    "start_time",
    "end_time",
    "label",
    "binary_label",
    "lateralization",
    "localization",
    "vigilance",
    "seizure_onset_sec",
    "seizure_duration_sec",
]

print(sz_meta[cols].head(20))

      patient_id          run_id  window_idx  start_time  end_time  \
11594    sub-001  sub-001_run-03       11594     57970.0   57980.0   
11595    sub-001  sub-001_run-03       11595     57975.0   57985.0   
11596    sub-001  sub-001_run-03       11596     57980.0   57990.0   
11597    sub-001  sub-001_run-03       11597     57985.0   57995.0   
11598    sub-001  sub-001_run-03       11598     57990.0   58000.0   
11599    sub-001  sub-001_run-03       11599     57995.0   58005.0   
11600    sub-001  sub-001_run-03       11600     58000.0   58010.0   
11601    sub-001  sub-001_run-03       11601     58005.0   58015.0   
11602    sub-001  sub-001_run-03       11602     58010.0   58020.0   
11603    sub-001  sub-001_run-03       11603     58015.0   58025.0   
11604    sub-001  sub-001_run-03       11604     58020.0   58030.0   
11605    sub-001  sub-001_run-03       11605     58025.0   58035.0   
11606    sub-001  sub-001_run-03       11606     58030.0   58040.0   
11607    sub-001  su